<a href="https://colab.research.google.com/github/rhonaeloisa/bigdata_lab4/blob/lab4/big_data_lab4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.functions import format_number


spark = SparkSession.builder.appName('lab4').getOrCreate()

sc = spark.sparkContext


In [ ]:
#create schema for flood control csv
df_schema = StructType([
    StructField("MainIsland", StringType(), True),
    StructField("Region", StringType(), True),
    StructField("Province", StringType(), True),
    StructField("LegislativeDistrict", StringType(), True),
    StructField("Municipality", StringType(), True),
    StructField("DistrictEngineeringOffice", StringType(), True),
    StructField("ProjectID", StringType(), True),
    StructField("ProjectName", StringType(), True),
    StructField("TypeOfWork", StringType(), True),
    StructField("FundingYear", IntegerType(), True),
    StructField("ContractId", StringType(), True),
    StructField("ApprovedBudgetForContract", DoubleType(), True),
    StructField("ContractCost", DoubleType(), True),
    StructField("ActualCompletionDate", StringType(), True),
    StructField("Contractor", StringType(), True),
    StructField("ContractorCount", StringType(), True),
    StructField("StartDate", StringType(), True)
])

#Load CSV as a DataFrame
df = spark.read.csv("/content/dpwh_flood_control_projects_lab4.csv", header=True, schema=df_schema)

#df.show(5, truncate=False)

#data cleaning
#remove duplicate rows
duplicate_rows = (
    df.groupBy(df.columns)
      .count()
      .filter(col("count") > 1)
      .select("ProjectName", "count")
)

print("Duplicate rows in CSV file")
duplicate_rows.show(truncate=False)

cleaned_data = df.dropDuplicates()

check_duplicates = (
    cleaned_data.groupBy(cleaned_data.columns)
      .count()
      .filter(col("count") > 1)
      .select("ProjectName", "count")
)

print("Check Duplicate rows in CSV file after cleaning")
check_duplicates.show()

Duplicate rows in CSV file
+--------------------------------------------------------------------------------------------------------------------------+-----+
|ProjectName                                                                                                               |count|
+--------------------------------------------------------------------------------------------------------------------------+-----+
|Construction of Slope Protection Structure along Jct. San Benito-Sta. Monica Road, San Benito, Surigao Del Norte          |3    |
|Construction of Shore Protection Structure along Poblacion 2, Burgos, Surigao del Norte                                   |5    |
|Construction of Shore Protection,Brgy. Pamosaingan,Socorro, Surigao del Norte                                             |3    |
|Construction of Flood Control Structure, Nuevo Campo, San Benito, Surigao del Norte                                       |5    |
|Reconstruction/Rehabilitation of Seawall and Reclamatio

In [ ]:
#INSIGHT #1
#Show the top 5 contractors by number of projects handled

contractor_split = cleaned_data.withColumn(
    "Contractor",
    explode(split(col("Contractor"), "/"))
)

contractor_split = contractor_split.withColumn(
    "Contractor",
    trim(col("Contractor"))
)

top_contractors = (cleaned_data.groupBy("Contractor")
                    .agg(count("*").alias("NumOfProj"))
                    .orderBy(col("NumOfProj")
                    .desc())
                    .limit(5)
)
print("Top 5 contractors by number of projects")
top_contractors.show(truncate=False)

Top 5 contractors by number of projects
+------------------------------------------------------------------------+---------+
|Contractor                                                              |NumOfProj|
+------------------------------------------------------------------------+---------+
|LEGACY CONSTRUCTION CORPORATION (FORMERLY: LEGACY CONSTRUCTION)         |115      |
|QM BUILDERS                                                             |88       |
|ALPHA & OMEGA GEN. CONTRACTOR & DEVELOPMENT CORP.                       |84       |
|ST. TIMOTHY CONSTRUCTION CORPORATION                                    |83       |
|SUNWEST, INC. (FORMERLY: SUNWEST CONSTRUCTION & DEVELOPMENT CORPORATION)|67       |
+------------------------------------------------------------------------+---------+



In [ ]:
#INSIGHT #2
#Show the total budget per province of region V

regionV = cleaned_data.filter(col("Region") == "Region V")

budget_per_province = (regionV.groupBy("Province")
                      .agg(sum("ContractCost")
                      .alias("TotalBudget"))
                      .orderBy(col("TotalBudget")
                      .desc())
)

print("Total Budget per Province, specifically in Region V")
budget_per_province_formatted = budget_per_province.withColumn(
    "TotalBudget", format_number("TotalBudget", 0)
)

budget_per_province_formatted.show(truncate=False)

Total Budget per Province, specifically in Region V
+---------------+--------------+
|Province       |TotalBudget   |
+---------------+--------------+
|Camarines Sur  |17,502,065,545|
|Albay          |15,916,670,980|
|Sorsogon       |6,451,452,595 |
|Camarines Norte|4,359,904,004 |
|Masbate        |3,305,591,502 |
|Catanduanes    |1,826,313,170 |
+---------------+--------------+



In [ ]:
#INSIGHT #3
#Total Budget per year

budget_per_year = (
    cleaned_data.filter(col("FundingYear").isNotNull())
                .groupBy("FundingYear")
                .agg(sum("ContractCost").alias("TotalBudget"))
                .orderBy(col("FundingYear").desc())
)

budget_per_year_formatted = budget_per_year.withColumn(
    "TotalBudget", format_number("TotalBudget", 0)
)

print("Total Budget per Year")
budget_per_year_formatted.show(truncate=False)

Total Budget per Year
+-----------+---------------+
|FundingYear|TotalBudget    |
+-----------+---------------+
|2025       |1,450,632,964  |
|2024       |144,628,793,326|
|2023       |203,376,105,562|
|2022       |163,642,310,423|
|2021       |28,313,726,674 |
|2020       |2,281,209,747  |
|2019       |335,826,929    |
|2018       |734,761,555    |
+-----------+---------------+



In [ ]:
#parquet file
parquet_path = "/content/parquet"
cleaned_data.write.mode("overwrite").format("parquet").save(parquet_path)
